# Lab 1: Prompt Engineering for Food Deviation Triage

We use one synthetic case: **post-cook cooling deviation** for a ready-to-eat product.

**Demo flow**

Demo 1a: Food QA Behavior and Rules (`Slide: System Prompt`)

Demo 1b: Build Risk Context from Shop-floor Note (`Slide: User Prompt`, `Best Practice 1: ICIO`)

Demo 2: Create Diagnostic Signal Map (`Slide: Improve Output Robustness`)
   - 2a. Structured Output (Free-form vs JSON)
   - 2b. Check Conditions (Missing Evidence / `null`)
   - 2c. Structured Output with Pydantic

Demo 3: Generate Diagnostic Questions / Draft Deviation Narrative (`Slide: Prompting Techniques`)
   - 3a. No CoT vs CoT-style Structured Trace
   - 3b. Zero-shot vs Few-shot

Demo 4: Other LLM Configurations (`Slide: Same name`)
   - 4a. Temperature Setting: Deterministic vs Creativity
   - 4b. Reasoning Effort: Evidence Gating / Condition Check

(Take-home) IRIS-style Analysis / Validation Prompt Template


**Main idea:**
This lab shows how prompt engineering can turn an incomplete food deviation note into structured risk context, diagnostic questions, validation checks, and a clearer deviation narrative.


## 0. Setup

Choose the provider in cell 5 before running the notebook.

- `OpenAI API`
  - Paid-tier, pay-as-you-go.
  - Use this if you already have an OpenAI API key or want OpenAI models.
  - Details: [OpenAI pricing](https://platform.openai.com/docs/pricing), [OpenAI models](https://platform.openai.com/docs/models)

- `Groq API`
  - Free-tier option for trying the workshop without payment.
  - Groq hosts openly available / open-weight models.
  - Details: [Groq models](https://console.groq.com/docs/models), [Groq rate limits](https://console.groq.com/docs/rate-limits)
  - Example models in this notebook: `llama-3.1-8b-instant`, `llama-3.3-70b-versatile`, `gpt-oss-120b`

**How to choose**
- If you can pay or already have an OpenAI API key, set `PROVIDER = "openai"`.
- If you want the free-tier option, set `PROVIDER = "groq"`.

*(RPM = Requests Per Minute, TPM = Tokens Per Minute)*


Install Libraries First
- Install only one of the following libraries, depending on the provider you want to use:

In [ ]:
# %%capture
# !pip install -qU langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 1.9 MB/s eta 0:00:00


In [ ]:
# if using groq api (Free Tier) => don't run the cell above, run this block instead
!pip install -qU langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.9 MB/s eta 0:00:00


In [ ]:
import os
import json
import re
import time
import getpass

PROVIDER = "groq"  # CHANGE THIS LINE ONLY: openai or groq

if PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
    default_llm = ChatOpenAI(
        model="gpt-5-nano",
        temperature=0.6,
        max_retries=2,
    )
elif PROVIDER == "groq":
    from langchain_groq import ChatGroq
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")
    default_llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0.6,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
else:
    raise ValueError("PROVIDER must be 'openai' or 'groq'")

# Keep a short alias for the early system-prompt demo cells.
llm = default_llm

print(f"Using provider: {PROVIDER}")


Enter your OpenAI API key: ··········
Using provider: openai


Also run this helper function for parsing a string that contains JSON inside it into a Python object.

In [ ]:
def parse_json_from_llm(text):
    """Extract JSON from an LLM response that may include Markdown or short prose."""
    text = text.strip()
    fenced = re.search(r"```(?:json)?\s*(.*?)```", text, flags=re.S | re.I)
    if fenced:
        text = fenced.group(1).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = min([i for i in [text.find("{"), text.find("[")] if i != -1], default=-1)
        end_obj = text.rfind("}")
        end_arr = text.rfind("]")
        end = max(end_obj, end_arr)
        if start != -1 and end != -1 and end > start:
            return json.loads(text[start:end + 1])
        raise

For the standard demos, the notebook mainly uses `gpt-5-nano`.

## 1. Running case

The case is written as a short shop-floor note.

In [ ]:
# cooling deviation
case_paragraph = """
Ready-to-eat chicken salad batch B-0519-N was produced on Line 2 during the night shift.
The process step under review is post-cook cooling. Cooling started at 00:00 with a recorded core temperature of 60.0C.
The procedure target is to cool from 60C to 21C within 2 hours, then to 5C within 4 additional hours.
At 02:00, the recorded core temperature was 24.8C. At 03:00, it was 14.2C. At 06:00, it was 5.7C.
The operator note says the batch was loaded into the chiller 20 minutes later than planned.
The chiller log shows one door-open alarm at 01:15, but the alarm duration is not included in the note.
QC observation says there was no visible packaging defect and the label and lot code were verified.
The previous three batches on the same line had no cooling deviation.
QA supervisor must review the case. Product release, hold, quarantine, recall scope, and CCP deviation disposition require Food Safety owner approval.
""".strip()

print(case_paragraph)

Ready-to-eat chicken salad batch B-0519-N was produced on Line 2 during the night shift.
The process step under review is post-cook cooling. Cooling started at 00:00 with a recorded core temperature of 60.0C.
The procedure target is to cool from 60C to 21C within 2 hours, then to 5C within 4 additional hours.
At 02:00, the recorded core temperature was 24.8C. At 03:00, it was 14.2C. At 06:00, it was 5.7C.
The operator note says the batch was loaded into the chiller 20 minutes later than planned.
The chiller log shows one door-open alarm at 01:15, but the alarm duration is not included in the note.
QC observation says there was no visible packaging defect and the label and lot code were verified.
The previous three batches on the same line had no cooling deviation.
QA supervisor must review the case. Product release, hold, quarantine, recall scope, and CCP deviation disposition require Food Safety owner approval.


## DEMO 1a : System Prompt

Before starting, compare the same question with no system prompt, a very different system role prompt, and the Food QA system prompt.

In [ ]:
simple_query_test = "I found a cooling deviation in a ready-to-eat product. What should I do?"


`Without System Prompt`

In [ ]:
plain_msg = llm.invoke(simple_query_test)
print("=== WITHOUT SYSTEM PROMPT ===")
print(plain_msg.content)


=== WITHOUT SYSTEM PROMPT ===
Thanks for asking. Here’s a practical way to handle a cooling deviation for a ready-to-eat product. I’ve split it into two paths so you can respond appropriately whether you’re a consumer or a worker/manager in production.

If you’re a consumer (you found a cooling deviation on a ready-to-eat product you bought):
- Do not eat the product. If the product is already opened, discard it.
- Do not try to “fix” or reheat it to make it safe. Some ready-to-eat foods can be risky if cooling/holding steps were not met.
- Keep the packaging and receipt if possible, along with the lot/batch code and best-by date. This helps the company investigate.
- Report the issue to the seller or manufacturer, using any recall/incident contact on the package or the company website. Provide the lot code, production date, and where you bought it.
- Check for any recall notices from the company or your local food safety authority. Follow their instructions.
- If you feel unwell after

`With Incorrect Role -  System Prompt (Role = Travel Planner, Rule = .[Rule Theme]..)`

In [ ]:
system_prompt_x_role = """
# Role
- You are a travel planner who helps people organize trips, transport, and hotel bookings.

# General Rules
- You help with trip planning, route suggestions, booking coordination, and simple travel questions.
- You do not review food safety evidence or make QA disposition decisions.
- You keep answers practical and concise.
- If the user asks for food safety judgment, direct them to QA/Food Safety.
""".strip()


In [ ]:
ops_msg = llm.invoke([
    {"role": "system", "content": system_prompt_x_role},
    {"role": "user", "content": simple_query_test},
])
print("=== WITH SYSTEM PROMPT (TRAVEL PLANNER ROLE) ===")
print(ops_msg.content)


=== WITH SYSTEM PROMPT (TRAVEL PLANNER ROLE) ===
I can’t judge safety. Please escalate the issue to your QA/Food Safety team right away. In the meantime, here are practical steps:

- Do not consume the product. Quarantine it in a clearly marked, sealed container or bag.
- Gather product details: brand, product name, size, lot/batch number, expiration date, packaging date, storage conditions, and when you found the deviation.
- Preserve evidence: keep the packaging, take photos of the label, lot code, and the deviation.
- Document the incident: note who found it, where, and any symptoms or potential exposure.
- Notify the appropriate authority in your context:
  - In a workplace: alert QA/Quality Assurance, file a nonconformance report, and follow the recall/withdrawal SOP. Hold inventory from sale/distribution.
  - As a consumer: contact the retailer or manufacturer with the lot number and purchase details; follow their instructions and keep receipts. If your local authorities require 

`With System Prompt (Role = Food QA, Rule = .[Rule Theme]..)`

In [ ]:
system_prompt_food_QA = """
# Role
- You are a QA support assistant for food manufacturing.
# General Rules
- You help organize evidence, identify signals, suggest hypotheses, and draft questions for human review.
- You must not decide product release, hold, quarantine, recall scope, CCP disposition, or deviation closure.
- You must not invent lab results, timestamps, measurements, or regulatory conclusions.
- You must not introduce new pathogen names, legal standards, or regulatory conclusions unless they are supplied in the input.
- If evidence is insufficient, state what is missing and say that QA/Food Safety approval is required.
- Keep responses concise unless the user explicitly asks for a detailed report.
""".strip()

system_prompt = system_prompt_food_QA


In [ ]:
food_qa_msg = llm.invoke([
    {"role": "system", "content": system_prompt_food_QA},
    {"role": "user", "content": simple_query_test},
])
print("=== WITH FOOD QA SYSTEM PROMPT ===")
print(food_qa_msg.content)


=== WITH FOOD QA SYSTEM PROMPT ===
I can help outline the next steps. Final disposition (hold/recall) should come from QA/Food Safety. Here’s a practical investigation checklist you can follow or give to QA:

1) Immediate containment (per your site SOP)
- Segregate and quarantine the affected batch/units as directed by QA.
- Do not release any affected product until QA approves next actions.

2) Gather evidence and initial facts
- Product: code/lot, production date/time, POU/pack size, allergen info.
- Deviation description: exactly what cooling deviation occurred (e.g., failed to reach target temp within specified time, temperature not below threshold by X hours).
- Equipment and process: cooling method (blast chiller, storage cooler, ambient), setpoints, equipment ID, any alarms or fault messages, last calibration.
- Environment: line location, ambient temperature, airflow conditions.
- Quantity affected: number of units, packaging type, and whether any packaging shows defects.
- Ope

## DEMO 1b: Build Risk Context from Shop-floor Note (User Prompt : ICIO)

**Task:** extract the process, affected unit, risk signal, evidence, owner, and missing context from a messy case note.

**Prompt concept:** ICIO makes the task, context, input, and output expectation explicit.

This demo should show that ICIO is not “long prompt = better.”  
It is a simple structure for reducing ambiguity when turning field notes into usable risk context.

generate concise explanation of user prompt : before starting to naive user prompt, icio user prompt.

### Explaining the `User Prompts`: Naive vs. ICIO

Before diving into the outputs, let's understand the two types of prompts used:

*   **Naive Prompt (`user_naive_prompt`):** This is a straightforward, open-ended prompt that asks the model to analyze the case and provide general recommendations. It lacks specific instructions on output format or constraints on the model's decision-making.

*   **ICIO Prompt (`user_icio_prompt`):** This prompt follows a structured 'Instruction, Context, Input, Output' (ICIO) pattern. It clearly defines:
    *   **Instruction:** The specific task (e.g., 'Create a food process risk map').
    *   **Context:** The model's role, limitations, background knowledge (e.g., 'QA support only,' 'does not make product disposition decisions').
    *   **Input:** The data the model should use (the `case_paragraph`).
    *   **Output:** The exact structure and headings for the response, guiding the model to produce a highly organized and constrained answer.

Begin with a `user naive prompt`.

In [ ]:
user_naive_prompt = f"""
Analyze this food safety case and tell me what to do:
{case_paragraph}
"""

naive_msg = default_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_naive_prompt},
])
print("=== WITHOUT ICIO / WITHOUT CONSTRAINTS ===")
print(naive_msg.content)


=== WITHOUT ICIO / WITHOUT CONSTRAINTS ===
Here’s a concise QA-oriented analysis with the information needed to support the Food Safety owner review. It highlights signals, gaps, and the questions to ask, plus a practical data-gathering plan. It stops short of making disposition decisions.

1) What happened (short assessment)
- Cooling profile issue: The target is 60C → 21C within 2 hours, then 21C → 5C within 4 additional hours.
- Observed data:
  - 00:00: 60.0C (start of cooling)
  - 02:00: 24.8C (above 21C; target not met by 2 h)
  - 03:00: 14.2C (below 21C)
  - 06:00: 5.7C (just above the 5C target)
- Delays/interruptions:
  - Operator note: batch loaded into chiller 20 minutes later than planned (possible reduction of effective cooling window).
  - Chiller log: one door-open alarm at 01:15; duration not provided.
- Other observations:
  - QC: no visible packaging defect; label/lot code verified.
  - Prior three batches on Line 2 had no cooling deviation.

Signals of concern
- Prim

Then, `user_icio_prompt`

In [ ]:
user_icio_prompt = f"""
Instruction:
Create a food process risk map for this case.

Context:
QA support only. The AI organizes evidence but does not make product disposition decisions.

Input:
{case_paragraph}

Output:
Write a concise plain-text note with these headings:
- Process
- Affected unit
- Risk signal
- Evidence / data sources
- Decision owner
- Missing context
""".strip()

icio_msg = default_llm.invoke([
    {"role": "system", "content": system_prompt_food_QA},
    {"role": "user", "content": user_icio_prompt}
    ]
)

print("=== WITH ICIO / WITH CONSTRAINTS ===")
print(icio_msg.content)

=== WITH ICIO / WITH CONSTRAINTS ===
- Process
  - Post-cook cooling step for ready-to-eat chicken salad batch B-0519-N on Line 2 (night shift).
  - Target: cool 60C to 21C within 2 hours; then 21C to 5C within 4 additional hours.
  - Observed: cooling started 00:00 at 60.0C; batch loaded into chiller 20 minutes later than planned; door-open alarm on chiller at 01:15 (duration not recorded).
  - Temperature readings: 02:00 = 24.8C; 03:00 = 14.2C; 06:00 = 5.7C.
  - QC note: no visible packaging defect; label and lot code verified.
  - Historical context: previous three batches on Line 2 had no cooling deviation.

- Affected unit
  - Batch B-0519-N; Line 2; post-cook cooling step during night shift.

- Risk signal
  - Cooling target: 21C by 02:00 not met (02:00 reading 24.8C).
  - Cooling target: 5C by 06:00 not met (06:00 reading 5.7C; still above target).
  - Operational deviation: 20-minute delay in loading into chiller.
  - Potential heat gain during cooling: door-open alarm at 01:15

**Checkpoint**

Compare the outputs:

- Did ICIO organize the answer better?
   - ANS: better
- Did it avoid making final QA/Food Safety decisions?
   - ANS: For the naive prompt -- model tends to recommend several decisions e.g. "quarantine".
   - ANS: For the system prompt + ICIO prompt, the model stayed in a QA support role and avoided making final disposition decisions. It clearly stated that QA/Food Safety decisions require review and approval from the responsible owners.

## DEMO 2a: Create Diagnostic Signal Map (Free-form vs JSON)

**Task:** separate observed facts, signals, hypotheses, supporting evidence, and missing evidence.

**Prompt concept:** same diagnostic task, different output format.

Both prompts ask for the same diagnostic content.  
The difference is whether the result is easy for humans only, or also easy for code to parse and reuse in the next step.

In [ ]:
freeform_signal_prompt = f"""
Create a Diagnostic Signal Map for the cooling deviation.

Fields per row:
data_source, observed_fact, signal, possible_hypothesis, supporting_evidence, missing_evidence.

Write plain text bullets only. Do not return JSON.

Case:
{case_paragraph}
""".strip()

freeform_signal_msg = default_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": freeform_signal_prompt}
])

print("=== WITHOUT JSON / PLAIN TEXT OUTPUT ===")
print(freeform_signal_msg.content)

=== WITHOUT JSON / PLAIN TEXT OUTPUT ===
- data_source: Operator production note
  observed_fact: Batch B-0519-N loaded into chiller 20 minutes later than planned
  signal: Pre-cooling warm hold potentially affected cooling window
  possible_hypothesis: The 20-minute loading delay reduced effective cooling time; early warm period plus later door-open event may have altered cooling trajectory
  supporting_evidence: Cooling started at 00:00 with 60.0C; operator note of 20-minute delay; 01:15 door-open alarm; 02:00 core Temp = 24.8C
  missing_evidence: Exact planned load time; duration of loading delay; ambient room temperature; chiller setpoints during early cooling

- data_source: Chiller log
  observed_fact: door-open alarm at 01:15; alarm duration not provided
  signal: Potential heat gain during door-open event impacting early cooling
  possible_hypothesis: Ingress of warm air or interruption of cooling cycle could have slowed initial cooling
  supporting_evidence: 01:15 door-open al

In [ ]:
signal_map_prompt = f"""
Create a Diagnostic Signal Map for the cooling deviation.

Use these fields:
data_source, observed_fact, signal, possible_hypothesis, supporting_evidence, missing_evidence.

Case:
{case_paragraph}

Return valid JSON:
{{
  "diagnostic_signal_map": [
    {{
      "data_source": "string",
      "observed_fact": "string",
      "signal": "string",
      "possible_hypothesis": "string",
      "supporting_evidence": "string",
      "missing_evidence": "string"
    }}
  ]
}}
""".strip()

signal_map_msg = default_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": signal_map_prompt}
    ]
)

print("=== WITH STRUCTURED JSON ===")
print(signal_map_msg.content)

signal_map = parse_json_from_llm(signal_map_msg.content)

=== WITH STRUCTURED JSON ===
{
  "diagnostic_signal_map": [
    {
      "data_source": "Cooling process data (batch B-0519-N, Line 2, night shift): cooling logs, operator notes, chiller log, QC observations, and line history.",
      "observed_fact": "Cooling started at 00:00 with core temperature 60.0C. Chiller door-open alarm recorded at 01:15 (duration not provided). At 02:00, core temperature was 24.8C; at 03:00, 14.2C; at 06:00, 5.7C. Batch loaded into chiller 20 minutes later than planned. Target of reaching 21C within 2 hours not met.",
      "signal": "Door-open event during cooling with schedule delay contributing to cooling timeline deviation.",
      "possible_hypothesis": "1) Door-open event at 01:15 caused heat ingress, slowing initial cooling and preventing reaching 21C by 2 hours. 2) 20-minute delay in loading into the chiller reduced effective cooling time, exacerbating the deviation. 3) Other factors (batch load density, chiller performance) could contribute but are no

**Checkpoint**

- Is the same key information easier to find in the JSON version?
  - ANS: Yes
- Why is JSON better?
  - ANS: It is easier to parse and reuse for downstream tasks.

## DEMO 2b: Check Conditions (Missing Evidence)

**Task:** extract known and unknown review fields from the case when some values are missing.

**Prompt concept:** "Check Conditions" means the model should follow the conditions we already know and return `null` for values that are not explicitly available, instead of trying to guess.

Here the missing fields are real gaps in the case, such as alarm duration, lab result, final disposition, and Food Safety approval.

- Free-form output may say: `not provided`, `unknown`, `requires review`
- JSON with a null rule should return: `null`, which becomes Python `None`


In [ ]:
boundary_fields = [
    "batch_id",
    "process_step",
    "temperature_at_02_00_c",
    "temperature_at_06_00_c",
    "alarm_duration_minutes",
    "lab_result_summary",
    "final_disposition_decision",
    "food_safety_owner_approval_status",
    "corrective_action_status",
]

boundary_field_list = "\n".join(f"- {field}" for field in boundary_fields)

boundary_base_prompt = f"""
Extract the following fields from the case:
{boundary_field_list}

Case:
{case_paragraph}
""".strip()

In [ ]:
free_boundary_prompt = f"""
{boundary_base_prompt}

Return plain English text only.
""".strip()


free_boundary_msg = default_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": free_boundary_prompt}
])

print("=== FREE-FORM MISSING VALUE OUTPUT ===")
print(free_boundary_msg.content)

=== FREE-FORM MISSING VALUE OUTPUT ===
batch_id: B-0519-N
process_step: post-cook cooling
temperature_at_02_00_c: 24.8
temperature_at_06_00_c: 5.7
alarm_duration_minutes: Not specified in the note (door-open alarm at 01:15; duration not provided)
lab_result_summary: Not provided
final_disposition_decision: Not determined; awaiting QA supervisor review and Food Safety owner approval
food_safety_owner_approval_status: Pending
corrective_action_status: Not documented


Let's try JSON output with `null` condition.

In [ ]:
json_null_boundary_prompt = f"""
{boundary_base_prompt}

Return valid JSON:
{{
  "boundary_condition_extract": {{
    "batch_id": "string or null",
    "process_step": "string or null",
    "temperature_at_02_00_c": "number or null",
    "temperature_at_06_00_c": "number or null",
    "alarm_duration_minutes": "number or null",
    "lab_result_summary": "string or null",
    "final_disposition_decision": "string or null",
    "food_safety_owner_approval_status": "string or null",
    "corrective_action_status": "string or null"
  }}
}}

Note:
If a field is not explicitly available in the case, return null.
""".strip()

json_null_boundary_msg = default_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": json_null_boundary_prompt}
])

print("\n=== JSON null MISSING VALUE OUTPUT ===")
print(json_null_boundary_msg.content)

boundary_extract = parse_json_from_llm(json_null_boundary_msg.content)
boundary_values = boundary_extract["boundary_condition_extract"]

print("\n=== PARSED PYTHON VALUES ===")
for field, value in boundary_values.items():
    print(f"{field}: {value!r} | python_type={type(value).__name__}")

none_fields = [field for field, value in boundary_values.items() if value is None]
print("\nFields parsed as Python None:", none_fields)


=== JSON null MISSING VALUE OUTPUT ===
{
  "boundary_condition_extract": {
    "batch_id": "B-0519-N",
    "process_step": "post-cook cooling",
    "temperature_at_02_00_c": 24.8,
    "temperature_at_06_00_c": 5.7,
    "alarm_duration_minutes": null,
    "lab_result_summary": "QC observation: no visible packaging defect; label and lot code verified.",
    "final_disposition_decision": null,
    "food_safety_owner_approval_status": null,
    "corrective_action_status": null
  }
}

=== PARSED PYTHON VALUES ===
batch_id: 'B-0519-N' | python_type=str
process_step: 'post-cook cooling' | python_type=str
temperature_at_02_00_c: 24.8 | python_type=float
temperature_at_06_00_c: 5.7 | python_type=float
alarm_duration_minutes: None | python_type=NoneType
lab_result_summary: 'QC observation: no visible packaging defect; label and lot code verified.' | python_type=str
final_disposition_decision: None | python_type=NoneType
food_safety_owner_approval_status: None | python_type=NoneType
corrective_a

## DEMO 2c: Structured Output with Pydantic

**Prompt concept:** instead of using a long prompt asking for JSON text only, bind the expected schema to the model call.

The schema uses `Optional[...]`, so missing fields can become `None`.

In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

class BoundaryConditionExtract(BaseModel):
    batch_id: Optional[str] = Field(description="Batch ID")
    process_step: Optional[str] = Field(description="Process step if explicitly available")
    temperature_at_02_00_c: Optional[float] = Field(description="02:00 core temperature")
    temperature_at_06_00_c: Optional[float] = Field(description="06:00 core temperature")
    alarm_duration_minutes: Optional[float] = Field(description="Alarm duration")
    lab_result_summary: Optional[str] = Field(description="Lab result summary")
    final_disposition_decision: Optional[str] = Field(description="Final product disposition")
    food_safety_owner_approval_status: Optional[str] = Field(description="Approval status")
    corrective_action_status: Optional[str] = Field(description="Corrective action status")


class BoundaryConditionOutput(BaseModel):
    boundary_condition_extract: BoundaryConditionExtract

new_json_null_boundary_prompt = f"""
Extract boundary-condition fields from this case. Use None for unavailable fields.\n\n{case_paragraph}
{boundary_base_prompt}

Note:
If a field is not explicitly available in the case, return null to that fields.
""".strip()

structured_boundary_llm = default_llm.with_structured_output(BoundaryConditionOutput)

structured_boundary_result = structured_boundary_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": new_json_null_boundary_prompt}
])

print("=== PYDANTIC STRUCTURED OUTPUT ===")
print(structured_boundary_result)

=== PYDANTIC STRUCTURED OUTPUT ===
boundary_condition_extract=BoundaryConditionExtract(batch_id='B-0519-N', process_step='post-cook cooling', temperature_at_02_00_c=24.8, temperature_at_06_00_c=5.7, alarm_duration_minutes=None, lab_result_summary=None, final_disposition_decision=None, food_safety_owner_approval_status=None, corrective_action_status=None)


In [ ]:
structured_boundary_result.model_dump_json()

'{"boundary_condition_extract":{"batch_id":"B-0519-N","process_step":"post-cook cooling","temperature_at_02_00_c":24.8,"temperature_at_06_00_c":5.7,"alarm_duration_minutes":null,"lab_result_summary":null,"final_disposition_decision":null,"food_safety_owner_approval_status":null,"corrective_action_status":null}}'

## DEMO 3a: Generate Diagnostic Questions (Chain-of-Thought Prompting)


**Prompt concept:** explicitly instruct the model how to reason.

In this demo, the prompt tells the model the reasoning steps directly:
`data seen -> signal observed -> hypothesis to test -> evidence needed -> decision relevance -> diagnostic question`.

This is different from Few-shot Chain-of-Thought in DEMO 4c, where the reasoning pattern is shown by examples instead of being explained mainly as rules.


In [ ]:
no_cot_prompt = f"""
Generate exactly 5 diagnostic questions for QA and Production.

Do not show reasoning. Return final questions only.

Case:
{case_paragraph}

Diagnostic Signal Map:
{json.dumps(signal_map, ensure_ascii=False, indent=2)}

Return valid JSON:
{{
  "prompt_style": "no_cot",
  "diagnostic_questions": [
    {{
      "question_id": "Q1",
      "target_role": "QA or Production or QA+Production",
      "diagnostic_question": "string"
    }}
  ],
  "summary": "one short sentence"
}}
""".strip()

no_cot_msg = default_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": no_cot_prompt}
])

print("=== NO COT: FINAL QUESTIONS ONLY ===")
print(no_cot_msg.content)

no_cot_output = parse_json_from_llm(no_cot_msg.content)


=== NO COT: FINAL QUESTIONS ONLY ===
{
  "prompt_style": "no_cot",
  "diagnostic_questions": [
    {
      "question_id": "Q1",
      "target_role": "QA or Production or QA+Production",
      "diagnostic_question": "What was the exact duration of the 01:15 door-open alarm, and was the chiller door left open during cooling? If yes, provide the total door-open time and the impacted interval."
    },
    {
      "question_id": "Q2",
      "target_role": "QA or Production or QA+Production",
      "diagnostic_question": "What was the exact time the batch was loaded into the chiller relative to the planned schedule, and how does this align with the cooling start time and the 2-hour target (21C) window?"
    },
    {
      "question_id": "Q3",
      "target_role": "QA or Production or QA+Production",
      "diagnostic_question": "Provide the chiller performance data for Batch B-0519-N from 00:00 to 06:00, including setpoints, actual temperatures, airflow, compressor status, and any alarms."
 

In [ ]:
cot_prompt = f"""
Generate diagnostic questions for QA and Production.

Case:
{case_paragraph}

Diagnostic Signal Map:
{json.dumps(signal_map, ensure_ascii=False, indent=2)}

For each question, show a short structured reasoning trace using this path:
1. what_data_seen: the exact supplied data, record, or observation used
2. what_signal_observed: the pattern, deviation, anomaly, or weak signal noticed
3. hypothesis_to_test: a possible explanation to investigate, not a final root cause
4. evidence_needed: concrete record, measurement, interview, or log needed to test the hypothesis
5. decision_relevance: why this question helps QA/Production narrow the investigation
6. diagnostic_question: the actual question to ask humans

Constraints:
- Do not conclude final root cause.
- Do not recommend release, hold, quarantine, or recall.
- Use only supplied case data and the supplied signal map.
- Keep every field concise: 1 short sentence each.
- Each question must be record-checkable or follow-up-checkable.
- Each question must connect clearly to one signal or evidence gap.
- Do not add pathogen names, legal standards, or regulatory conclusions not supplied in the input.
- Return exactly 5 reasoning traces.

Return valid JSON with exactly this schema:
{{
  "prompt_style": "cot_structured_trace",
  "question_reasoning_traces": [
    {{
      "question_id": "Q1",
      "target_role": "QA or Production or QA+Production",
      "what_data_seen": "string",
      "what_signal_observed": "string",
      "hypothesis_to_test": "string",
      "evidence_needed": "string",
      "decision_relevance": "string",
      "diagnostic_question": "string"
    }}
  ],
  "summary": "one short sentence describing how the trace improves auditability"
}}
""".strip()

cot_msg = default_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": cot_prompt}
])
print("=== COT: STRUCTURED REASONING TRACE ===")
print(cot_msg.content)

cot_output_round1 = parse_json_from_llm(cot_msg.content)

# Use the CoT trace directly in downstream analysis.
diagnostic_questions = cot_output_round1


=== COT: STRUCTURED REASONING TRACE ===
{
  "prompt_style": "cot_structured_trace",
  "question_reasoning_traces": [
    {
      "question_id": "Q1",
      "target_role": "QA+Production",
      "what_data_seen": "Cooling process data shows a door-open alarm at 01:15 with no duration provided.",
      "what_signal_observed": "Door-open event suggesting heat ingress may have slowed initial cooling.",
      "hypothesis_to_test": "The door-open duration could have caused heat ingress slowing cooling, affecting reaching 21C by 2h.",
      "evidence_needed": "Exact duration of the 01:15 door-open alarm and any additional door openings during 00:00–02:00 from the chiller log.",
      "decision_relevance": "Clarifies if a door breach affected cooling rate, guiding corrective actions.",
      "diagnostic_question": "What was the exact duration of the 01:15 door-open alarm, and were there any other door openings from 00:00 to 02:00 on Line 2 during cooling of batch B-0519-N?"
    },
    {
      

**Why is CoT useful?**

- Answer: The AI response can be checked more easily because it includes intermediate reasoning steps.
- Answer: It can sometimes improve answer quality by encouraging the model to reason more carefully, although it may make the response slower.

## DEMO 3b: Draft Deviation Narrative (Zero-shot vs Few-shot Prompting)

**Task:** turn validated analysis into a concise deviation narrative / non-conformance draft.

**Prompt concept:** use an example to control output pattern.

- Zero-shot prompt: flexible but inconsistent
- Few-shot: controls pattern by example

Here, the new concept is few-shot prompting: give the model one good narrative example, then ask it to follow that pattern.

Let's start from zero-shot prompting.

In [ ]:
zero_shot_narrative_prompt = f"""
Write a deviation report for this cooling deviation case.

Case:
{case_paragraph}

Useful context:
- Diagnostic Signal Map:
{json.dumps(signal_map, ensure_ascii=False, indent=2)}
- Diagnostic Questions:
{json.dumps(diagnostic_questions, ensure_ascii=False, indent=2)}

Use the case facts only and keep the output concise and structured.
""".strip()

zero_shot_narrative_msg = default_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": zero_shot_narrative_prompt}
])

print("=== ZERO-SHOT PROMPT ===")
print(zero_shot_narrative_msg.content)


=== ZERO-SHOT PROMPT ===
Deviation Report: Cooling Deviation for B-0519-N (Line 2, Night Shift)

1) Case identifiers
- Product: Ready-to-eat chicken salad, batch B-0519-N
- Location: Line 2
- Timeframe: Night shift cooling start 00:00
- Process step: Post-cook cooling
- Reported by: QA review requested

2) Deviation description and timeline
- Cooling target: Cool from 60°C to 21°C within 2 hours, then to 5°C within 4 additional hours (i.e., by ~06:00)
- Observed data:
  - 00:00: Core temp 60.0°C (start of cooling)
  - 01:15: Chiller door-open alarm recorded (duration not provided)
  - 02:00: Core temp 24.8°C
  - 03:00: Core temp 14.2°C
  - 06:00: Core temp 5.7°C
- Operational note: Batch loaded into the chiller 20 minutes later than planned
- Packaging/Label: QC observation – no visual packaging defect; label and lot code verified
- Line history: Previous three batches on Line 2 had no cooling deviation

3) Observations and signals
- Primary signal: Cooling did not meet the 2-hour targ

### Apply Narrative Pattern (Few-shot)

**Prompt concept:** show one good example, then ask the model to follow the same pattern.

In [ ]:
few_shot_prompt = f"""
Example input:
A packaging line found three cartons from lot FG-204 with an older label artwork. The old label omitted one allergen statement. Shipment had not started. QA label reconciliation was opened.

Example good narrative:
Observed facts: Three cartons from lot FG-204 were found with older label artwork before shipment.
Affected unit: Finished goods lot FG-204 on the packaging line.
Evidence reviewed: operator observation and QA label reconciliation start record.
Hypotheses under review: label roll-up mix-up or incomplete line clearance. These are hypotheses only.
Missing evidence: full label reconciliation, line clearance record, count of affected units.
Approval required: QA/Food Safety must approve label disposition, rework, release, or customer notification.

Now draft a deviation narrative for the cooling deviation case using the same structure.

Cooling deviation case:
{case_paragraph}

Useful context:
- Diagnostic Signal Map:
{json.dumps(signal_map, ensure_ascii=False, indent=2)}
- Diagnostic Questions:
{json.dumps(diagnostic_questions, ensure_ascii=False, indent=2)}

Keep the same six-line structure as the example.
Do not finalize root cause or product disposition.
""".strip()

few_shot_msg = default_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": few_shot_prompt}
])

print("=== FEW-SHOT PROMPT ===")
print(few_shot_msg.content)


=== FEW-SHOT PROMPT ===
Observed facts: Ready-to-eat chicken salad batch B-0519-N on Line 2 during the night shift; post-cook cooling started at 00:00 with core temperature 60.0C; target is to reach 21C within 2 hours, then 5C within 6 total hours. At 02:00 core temp was 24.8C; at 03:00 14.2C; at 06:00 5.7C. The operator note states the batch was loaded into the chiller 20 minutes later than planned. The chiller log shows a door-open alarm at 01:15, but duration is not provided. QC observation says there was no visible packaging defect and the label and lot code were verified. The previous three batches on Line 2 had no cooling deviation.

Affected unit: Finished goods batch B-0519-N undergoing post-cook cooling on Line 2.

Evidence reviewed: Operator cooling observations; chiller log showing a door-open alarm at 01:15 (duration not provided); operator note of a 20-minute loading delay; QC observation confirming no packaging defect and verified label/lot code; line history indicating n

Use few-shot prompting when you want the output to follow an exact pattern.

In this demo, the example mainly controls **style, tone, and format** of the deviation narrative.


## DEMO 3c: Few-shot Chain-of-Thought

**Task:** generate diagnostic questions by imitating reasoning trace examples.

**Prompt concept:** examples can guide the model's reasoning pattern.

- DEMO 4a: instruct the reasoning steps directly.
- DEMO 4b: use few-shot examples to guide style, tone, and format.
- DEMO 4c: use few-shot reasoning trace examples to guide how the model thinks through a diagnostic question.

Key idea: **shows rather than tells**.  
Instead of writing a long instruction for how to reason, we show examples of reasoning traces and ask the model to follow the same pattern.


### Reasoning by Example

Each example has:

- `Input`: a short food deviation case
- `Reasoning Trace Example`: step-by-step diagnostic thinking

There is no full output example here because the purpose is to show **Few-shot CoT**, not another output-format example.

The model should imitate the reasoning trace pattern and apply it to the cooling deviation case.


In [ ]:
few_shot_cot_prompt = f"""
You are helping QA and Production generate diagnostic questions for a food safety deviation investigation.

Follow the reasoning trace style shown in the examples.

Example 1

Input:
A packaging line found three cartons from lot FG-204 with older label artwork.
The old label omitted one allergen statement.
Shipment had not started.
QA label reconciliation was opened.

Reasoning Trace Example:
Step 1: I see that three cartons from lot FG-204 used older label artwork before shipment.
Step 2: This suggests a label version mismatch affecting finished goods.
Step 3: A hypothesis to test is label roll-up mix-up or incomplete line clearance.
Step 4: Evidence still missing includes the full label reconciliation result, line clearance record, and total affected carton count.
Step 5: Therefore, the first diagnostic question should ask QA to confirm label reconciliation, line clearance, and affected quantity.

Example 2

Input:
A metal detector rejected two packs during routine production.
The recheck result was inconsistent.
One pack passed on repeat testing, but one pack failed again.
The detector sensitivity setting was changed after maintenance earlier that morning.

Reasoning Trace Example:
Step 1: I see that two packs were rejected, and one pack failed again during repeat testing.
Step 2: This suggests either a possible foreign material issue or an unstable detection process.
Step 3: A hypothesis to test is true contamination, detector instability, or incorrect sensitivity setting after maintenance.
Step 4: Evidence still missing includes detector challenge test records, maintenance record, sensitivity setting history, and rejected pack inspection result.
Step 5: Therefore, the first diagnostic question should ask QA and Engineering to confirm detector challenge tests, maintenance details, settings, and inspection results.

Now apply the same reasoning trace style to this cooling deviation case.

Input:
{case_paragraph}

Diagnostic Signal Map:
{json.dumps(signal_map, ensure_ascii=False, indent=2)}

Return valid JSON only:
{{
  "prompt_style": "few_shot_chain_of_thought",
  "diagnostic_reasoning_questions": [
    {{
      "question_id": "Q1",
      "target_role": "QA or Production or QA+Production",
      "reasoning_trace": [
        "Step 1: ...",
        "Step 2: ...",
        "Step 3: ...",
        "Step 4: ...",
        "Step 5: ..."
      ],
      "diagnostic_question": "..."
    }}
  ],
  "summary": "one short sentence explaining what the questions focus on"
}}

Constraints:
- Generate exactly 3 diagnostic reasoning questions.
- Use only supplied facts.
- Do not conclude final root cause.
- Do not decide product release, hold, quarantine, recall scope, or deviation closure.
- Keep hypotheses as hypotheses only.
- Make missing evidence explicit.
- Keep each reasoning trace concise and traceable.
""".strip()

few_shot_cot_msg = default_llm.invoke([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": few_shot_cot_prompt}
])

print("=== FEW-SHOT COT: REASONING TRACE BY EXAMPLE ===")
print(few_shot_cot_msg.content)

few_shot_cot_output = parse_json_from_llm(few_shot_cot_msg.content)


=== FEW-SHOT COT: REASONING TRACE BY EXAMPLE ===
{
  "prompt_style": "few_shot_chain_of_thought",
  "diagnostic_reasoning_questions": [
    {
      "question_id": "Q1",
      "target_role": "QA or Production or QA+Production",
      "reasoning_trace": [
        "Step 1: Cooling started at 00:00 with core temperature 60.0C; a door-open alarm is recorded at 01:15, but the duration is not provided.",
        "Step 2: A door-open event during cooling can cause heat ingress, potentially slowing the cooling rate and affecting the ability to reach target temperatures on time.",
        "Step 3: A hypothesis to test is that the 01:15 door-open event (and any additional door openings) contributed to heat gain and slowed cooling in the 00:00–02:00 window.",
        "Step 4: Evidence missing includes the exact duration of the 01:15 alarm, whether there were additional door openings, and the full door-open timeline correlated to cooling data.",
        "Step 5: Therefore, the first diagnostic ques

**Teaching point**

DEMO 4c is different from DEMO 4a.

- In DEMO 4a, we tell the model how to reason by writing explicit reasoning steps.
- In DEMO 4c, we show examples of reasoning traces.

The model learns the pattern from examples:

`Input -> Step 1 -> Step 2 -> Step 3 -> Step 4 -> Step 5 -> Diagnostic question`

This is Few-shot Chain-of-Thought: examples guide the reasoning path, not only the final answer style.


## DEMO 4a: Deterministic vs Creativity (Temperature Setting)

**Task:** compare how much variation appears when the model writes internal email subject lines about the same event.

**Prompt concept:** low temperature should stay close to one stable wording; high temperature should produce more varied, creative phrasing.

**Provider note**
- OpenAI users: this demo uses `gpt-4o-mini`.
- Groq users: this demo uses `openai/gpt-oss-120b` with the same temperature settings.

Write 3 alternative internal email subject lines for the QA team about this cooling deviation.

- `temperature=0`
- `temperature=1`

Guidelines:
- Preserve the facts from the case.
- Vary wording, tone, and structure.
- Each subject line should be short and concise.
- Do not ask diagnostic questions.
- Do not judge product disposition.

Learners inspect whether the outputs repeat, vary, or drift.


In [ ]:
system_prompt_food_QA = """
# Role
- You are a QA support assistant for food manufacturing.
# General Rules
- You help organize evidence, identify signals, suggest hypotheses, and draft questions for human review.
- You must not decide product release, hold, quarantine, recall scope, CCP disposition, or deviation closure.
- You must not invent lab results, timestamps, measurements, or regulatory conclusions.
- You must not introduce new pathogen names, legal standards, or regulatory conclusions unless they are supplied in the input.
- If evidence is insufficient, state what is missing and say that QA/Food Safety approval is required.
- Keep responses concise unless the user explicitly asks for a detailed report.
""".strip()

system_prompt = system_prompt_food_QA


In [ ]:
# cooling deviation
case_paragraph = """
Ready-to-eat chicken salad batch B-0519-N was produced on Line 2 during the night shift.
The process step under review is post-cook cooling. Cooling started at 00:00 with a recorded core temperature of 60.0C.
The procedure target is to cool from 60C to 21C within 2 hours, then to 5C within 4 additional hours.
At 02:00, the recorded core temperature was 24.8C. At 03:00, it was 14.2C. At 06:00, it was 5.7C.
The operator note says the batch was loaded into the chiller 20 minutes later than planned.
The chiller log shows one door-open alarm at 01:15, but the alarm duration is not included in the note.
QC observation says there was no visible packaging defect and the label and lot code were verified.
The previous three batches on the same line had no cooling deviation.
QA supervisor must review the case. Product release, hold, quarantine, recall scope, and CCP deviation disposition require Food Safety owner approval.
""".strip()

print(case_paragraph)

In [ ]:
signal_map = {
  "diagnostic_signal_map": [
    {
      "data_source": "Cooling process data (batch B-0519-N, Line 2, night shift): cooling logs, operator notes, chiller log, QC observations, and line history.",
      "observed_fact": "Cooling started at 00:00 with core temperature 60.0C. Chiller door-open alarm recorded at 01:15 (duration not provided). At 02:00, core temperature was 24.8C; at 03:00, 14.2C; at 06:00, 5.7C. Batch loaded into chiller 20 minutes later than planned. Target of reaching 21C within 2 hours not met.",
      "signal": "Door-open event during cooling with schedule delay contributing to cooling timeline deviation.",
      "possible_hypothesis": "1) Door-open event at 01:15 caused heat ingress, slowing initial cooling and preventing reaching 21C by 2 hours. 2) 20-minute delay in loading into the chiller reduced effective cooling time, exacerbating the deviation. 3) Other factors (batch load density, chiller performance) could contribute but are not evidenced.",
      "supporting_evidence": "01:15 door-open alarm; 02:00 temp 24.8C (above 21C); 03:00 14.2C; 06:00 5.7C; operator note of a 20-minute loading delay; QC verified packaging; prior three batches on the line had no cooling deviation.",
      "missing_evidence": "Duration of the door-open alarm; exact start time of cooling relative to loading; batch weight/volume; chiller performance data during interval (setpoints, airflow, compressor status); ambient conditions; any additional door openings or disruptions; exact end time for cooling."
    },
    {
      "data_source": "Cooling process data (batch B-0519-N) and operator/line history indicating loading schedule.",
      "observed_fact": "Operator reported the batch was loaded into the chiller 20 minutes later than planned. Measured temps show 02:00 = 24.8C, 03:00 = 14.2C, 06:00 = 5.7C; 21C target not met within 2 hours; 5C reached at about 6 hours. QC packaging verified; no visible defects; prior batches on Line 2 had no cooling deviation.",
      "signal": "Delayed chiller loading as a contributor to cooling deviation.",
      "possible_hypothesis": "The 20-minute loading delay reduced effective cooling time, helping explain the missed 2-hour target; potential interaction with any door-open event (if occurred) could have compounded the issue. No packaging defects detected.",
      "supporting_evidence": "20-minute delay noted; cooling temps indicate slower initial cooling relative to target; 21C by 2 hours not achieved; packaging and labeling OK; previous batches were normal.",
      "missing_evidence": "Exact time when cooling began relative to loading; batch weight/volume; chiller performance data during cooling (setpoints, airflow, temperature logs); ambient conditions; any other disruptions or door openings."
    }
  ]
}

In [ ]:
temperature_prompt = f"""
Write 3 alternative internal email subject lines for the QA team about this cooling deviation.

Case:
{case_paragraph}

Diagnostic Signal Map:
{json.dumps(signal_map, ensure_ascii=False, indent=2)}

Instructions:
- Preserve the facts from the case.
- Vary wording, tone, and structure.
- Each subject line should be short and concise.
- Do not ask diagnostic questions.
- Do not judge product disposition.
- Return a concise numbered list only.
""".strip()

temperature_results = {}

for temp in [0, 1]:
    if PROVIDER == "openai":
        temp_llm = ChatOpenAI(
            model="gpt-4o-mini",
            temperature=temp,
            max_retries=2,
        )
    elif PROVIDER == "groq":
        temp_llm = ChatGroq(
            model="openai/gpt-oss-120b",
            temperature=temp,
            max_retries=2,
        )
    else:
        raise ValueError("PROVIDER must be 'openai' or 'groq'")

    temperature_results[temp] = []

    print("\n" + "=" * 90)
    print(f"TEMPERATURE = {temp} | 3 RUNS")
    print("=" * 90)

    for round_id in range(1, 4):
        temp_msg = temp_llm.invoke([
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": temperature_prompt}
        ])

        temperature_results[temp].append({
            "round": round_id,
            "content": temp_msg.content,
        })

        print(f"\n--- Round {round_id} ---")
        print(temp_msg.content)
        time.sleep(1)



TEMPERATURE = 0 | 3 RUNS

--- Round 1 ---
1. Review Required: Cooling Deviation for Batch B-0519-N on Line 2  
2. Attention Needed: Post-Cook Cooling Issues for Chicken Salad Batch B-0519-N  
3. Urgent: Cooling Timeline Deviation Analysis for Batch B-0519-N

--- Round 2 ---
1. Review Required: Cooling Deviation for Batch B-0519-N on Line 2  
2. Attention Needed: Post-Cook Cooling Issues for Chicken Salad Batch B-0519-N  
3. Urgent: Cooling Timeline Deviation Analysis for Batch B-0519-N

--- Round 3 ---
1. Review Required: Cooling Deviation for Batch B-0519-N on Line 2  
2. Attention Needed: Post-Cook Cooling Issues for Chicken Salad Batch B-0519-N  
3. Urgent: Cooling Timeline Deviation Analysis for Batch B-0519-N

TEMPERATURE = 1 | 3 RUNS

--- Round 1 ---
1. Review Needed: Cooling Deviation for Batch B-0519-N on Line 2  
2. Urgent: Analysis of Cooling Delay for Night Shift Chicken Salad Batch  
3. Action Required: Deviation in Cooling Process for Batch B-0519-N

--- Round 2 ---
1. "R

Observation

- At `temperature = 0`, the model returns almost identical outputs across multiple runs.  
This is useful when we need stable and consistent responses.

- At `temperature = 1`, the model generates **more varied wording and structure across runs**.  
**The key facts are still preserved, but the phrasing becomes more diverse**.

In this example, a higher temperature is useful for creative wording, such as alternative email subject lines, while a lower temperature is better for consistency.

## DEMO 4b: Evidence Gating / Condition Check (Reasoning Effort)

**Remark:**  
- For OpenAI, this section uses `gpt-5-mini` with `reasoning_effort`.
- For Groq, this section uses `openai/gpt-oss-120b` with `reasoning_effort`.
- Temperature mainly controls output randomness, consistency, and variation.
- Reasoning effort controls how carefully the model checks evidence, contradictions, and missing information before answering.

**Task:** verify 5 statements against the case evidence.

**Prompt Concept:** reason carefully about supported / unsupported / unknown statements.

Let's configure reasoning effort to:

- `"effort": "low"`
- `"effort": "high"`

Then compare the differences in traceability, evidence citation, and abstaining when information is missing.


Signal map from previous steps is prepared for convenience during the demo.

In [ ]:
import json
import time

reasoning_effort_prompt = f"""
You are verifying statements about a food manufacturing deviation case.

Case:
{case_paragraph}

Diagnostic Signal Map:
{json.dumps(signal_map, ensure_ascii=False, indent=2)}

Task:
For each statement below, return:
- verdict: PASS / FAIL / UNKNOWN
- evidence: short quote or paraphrase from the case
- reason: one short sentence
- missing_data: what is needed if verdict is UNKNOWN

Statements:
1. Batch B-0519-N was loaded into the chiller 20 minutes later than planned.
2. The chiller door-open alarm duration at 01:15 is known.
3. The 02:00 core temperature is above the 21C target.
4. The same calibrated probe location was used for all recorded temperatures.
5. QA/Food Safety approval is required before any disposition decision.

Rules:
- Use supplied evidence only.
- Do not invent missing data.
- If evidence is insufficient, verdict must be UNKNOWN and say "ยังสรุปไม่ได้".
- Keep the answer concise and structured.
""".strip()

if PROVIDER == "openai":
    for effort in ["low", "high"]:
        reasoning_config = {
            "effort": effort,
            "summary": "auto",
        }
        reasoning_model = ChatOpenAI(
            model="gpt-5-mini",
            reasoning=reasoning_config,
        )
        start_time = time.time()
        response_reasoning = reasoning_model.invoke(reasoning_effort_prompt)
        elapsed = time.time() - start_time
        print("\n" + "=" * 90)
        print(f'REASONING EFFORT = "{effort}"')
        print("=" * 90)

        content = response_reasoning.content

        if isinstance(content, list):
            for item in content:
                if item.get("type") == "text":
                    print(item.get("text", "").strip())
        else:
            print(str(content).replace("  \n", "\n").strip())

        print(f"\nElapsed time: {elapsed:.2f} seconds")
        print()

elif PROVIDER == "groq":
    for effort in ["low", "high"]:
        reasoning_model = ChatGroq(
            model="openai/gpt-oss-120b",
            temperature=0.6,
            reasoning_effort=effort,
            max_retries=2,
        )
        start_time = time.time()
        response_reasoning = reasoning_model.invoke(reasoning_effort_prompt)
        elapsed = time.time() - start_time
        print("\n" + "=" * 90)
        print(f'REASONING EFFORT = "{effort}"')
        print("=" * 90)

        content = response_reasoning.content

        if isinstance(content, list):
            for item in content:
                if item.get("type") == "text":
                    print(item.get("text", "").strip())
        else:
            print(str(content).replace("  \n", "\n").strip())

        print(f"\nElapsed time: {elapsed:.2f} seconds")
        print()
else:
    raise ValueError("PROVIDER must be 'openai' or 'groq'")



REASONING EFFORT = "low"
1)
- verdict: PASS
- evidence: "The operator note says the batch was loaded into the chiller 20 minutes later than planned."
- reason: Operator note explicitly states a 20-minute loading delay.
- missing_data: N/A

2)
- verdict: FAIL
- evidence: "The chiller log shows one door-open alarm at 01:15, but the alarm duration is not included in the note."
- reason: Duration is explicitly not provided in the available records.
- missing_data: N/A

3)
- verdict: PASS
- evidence: "At 02:00, the recorded core temperature was 24.8C" and target is "reach 21C within 2 hours."
- reason: 24.8°C at 02:00 is above the 21°C target.
- missing_data: N/A

4)
- verdict: UNKNOWN (ยังสรุปไม่ได้)
- evidence: No information provided about probe location consistency or calibration across readings.
- reason: Case data does not state whether the same calibrated probe location was used for all measurements.
- missing_data: Probe location records, calibration logs, or measurement procedure 

## (Take-home) IRIS Analysis / Validation Prompt Template

Task: LLM is asked to use
- the risk map (a cleaned scope/context from the earlier process-risk mapping step, including process, risk, data sources, and decision constraints)
- and the shop-floor note (the raw incident note from production) to generate an IRIS-style analysis
-  then validate whether the analysis is evidence-grounded, traceable, free from unsupported assumptions, and within the human approval boundary.


In [ ]:
mock_input = """
Ready-to-eat chicken salad batch B-0519-N was produced on Line 2 during the night shift.
The process step under review is post-cook cooling. Cooling started at 00:00 with a recorded core temperature of 60.0C.
The procedure target is to cool from 60C to 21C within 2 hours, then to 5C within 4 additional hours.
At 02:00, the recorded core temperature was 24.8C. At 03:00, it was 14.2C. At 06:00, it was 5.7C.
The operator note says the batch was loaded into the chiller 20 minutes later than planned.
The chiller log shows one door-open alarm at 01:15, but the alarm duration is not included in the note.
QC observation says there was no visible packaging defect and the label and lot code were verified.
The previous three batches on the same line had no cooling deviation.
QA supervisor must review the case. Product release, hold, quarantine, recall scope, and CCP deviation disposition require Food Safety owner approval.
""".strip() ## Same case as previous demos

In [ ]:
mock_risk_map = {
    "process": "post-cook cooling",
    "product": "ready-to-eat chicken salad",
    "affected_unit": "batch B-0519-N, Line 2, night shift",
    "risk": "cooling temperature deviation",
    "data_sources": [
        "temperature readings",
        "operator note",
        "chiller door-open alarm",
        "QC observation",
        "previous batch history",
    ],
    "decision_owner": "QA supervisor review; Food Safety owner approval for disposition",
    "decision_constraints": [
        "AI must not decide release, hold, quarantine, recall, CCP disposition, or deviation closure",
        "AI must list missing evidence when evidence is insufficient",
    ]
}


We have two versions of the IRIS prompt: the version before improving the prompt structure and the version after improving the prompt structure.

### Original Prompt

In [ ]:
# ANALYSIS PROMPT
original_iris_analysis_prompt = f"""
คุณคือผู้ช่วยวิเคราะห์ข้อมูลในอุตสาหกรรมอาหาร

Context Block:
- กระบวนการ: {mock_risk_map["process"]}
- ปัญหาที่เลือก: {mock_risk_map["risk"]}
- หน่วยที่ได้รับผลกระทบ: {mock_risk_map["affected_unit"]}

Input Block:
{mock_input}

Check Condition Block:
- ห้ามสรุป Root Cause เกินหลักฐาน
- ห้ามเสนอ Release, quarantine หรือ Recall โดยไม่มี Human approval

งานที่ต้องการให้วิเคราะห์:
1. ระบุ signal ที่สำคัญ
2. ตั้งสมมติฐานที่เป็นไปได้
3. ระบุหลักฐานที่สนับสนุนแต่ละสมมติฐาน
4. ระบุหลักฐานที่ยังขาด
5. เสนอคำถามที่ควรถามทีม Production หรือ Quality Assurance เพิ่ม
""".strip()

analysis_from_AI = default_llm.invoke(original_iris_analysis_prompt)

# VALIDATION PROMPT
original_iris_validation_prompt = f"""
คุณคือผู้ช่วยตรวจสอบผลการวิเคราะห์ในอุตสาหกรรมอาหาร

Context Block:
- กระบวนการ: {mock_risk_map["process"]}
- ปัญหาที่เลือก: {mock_risk_map["risk"]}
- หน่วยที่ได้รับผลกระทบ: {mock_risk_map["affected_unit"]}

Input Block:
- AI-generated analysis: {analysis_from_AI.content}
- Decision constraints: {json.dumps(mock_risk_map["decision_constraints"], ensure_ascii=False, indent=2)}

Task:
1. คำตอบอ้างอิงข้อมูลจริงหรือไม่
2. ย้อนกลับไปที่ lot, batch, line, shift หรือเวลาได้ไหม
3. มีสมมติฐานใดที่ยังไม่มีหลักฐานสนับสนุน
4. มีความเสี่ยงด้าน food safety, compliance หรือ customer impact หรือไม่
5. จุดใดต้องให้มนุษย์อนุมัติก่อนดำเนินการ

Check Condition Block:
- หากหลักฐานไม่พอ ให้ระบุว่า “ยังสรุปไม่ได้” และบอกว่าต้องใช้ข้อมูลอะไรเพิ่ม
""".strip()

validation_from_AI = default_llm.invoke(original_iris_validation_prompt)

print("=== IRIS ANALYSIS OUTPUT ===")
print(analysis_from_AI.content)
print()
print("=== IRIS VALIDATION OUTPUT ===")
print(validation_from_AI.content)


=== IRIS ANALYSIS OUTPUT ===
ด้านล่างเป็นกรอบวิเคราะห์ที่สอดคล้องกับคำขอ โดยไม่สรุป Root Cause และไม่เสนอ Release/Quarantine/Recall โดยไม่ผ่านการอนุมัติจากเจ้าของความปลอดภัยอาหาร

1) signal สำคัญ (signal ที่ควรให้ความสำคัญในการประเมิน)
- จุดเริ่มต้นกระบวนการ: core temperature เริ่มต้นที่ 60.0 C ในช่วง post-cook cooling
- เป้าหมายการเย็น: ต้องลดจาก 60 C ไปยัง 21 C ภายใน 2 ชั่วโมง แล้วลดจาก 21 C ไป 5 C ภายใน 4 ชั่วโมงถัดไป
- ความล่าช้าในการโหลดเข้า chiller: เจ้าหน้าที่ระบุว่า batch ถูกโหลดเข้าเครื่องทำเย็นชิลเลอร์ช้ากว่าที่วางแผนไว้ 20 นาที
- เหตุการณ์ประตูห้องเย็นเปิด: มีบันทึกเตือน “door-open” เวลา 01:15 แต่ระยะเวลาที่เปิดประตูไม่ได้ระบุ
- แนวโน้มการเย็น: 02:00 core 24.8 C; 03:00 core 14.2 C; 06:00 core 5.7 C
- จุดเปรียบเทียบ: 3 บทนำก่อนหน้านี้ on-line บนสายเดียวกันไม่มี deviation ในการ cooling
- ตรวจสอบคุณภาพ/บรรจุภัณฑ์: ไม่มีข้อบกพร่องในการแพ็กเกจและโลหะ/รหัสล็อตถูกตรวจสอบแล้ว

2) สมมติฐานที่เป็นไปได้ (hypotheses) — บอกได้ว่าเป็นไปได้หลายแนวทางโดยไม่สรุป causes
- H1: ความล่าช้าในการโ

### Improved Prompt

In [ ]:
# IMPROVED ANALYSIS PROMPT

improved_iris_analysis_prompt = f"""
คุณคือผู้ช่วยวิเคราะห์ข้อมูลในอุตสาหกรรมอาหาร

Context Block:
- กระบวนการ: {mock_risk_map["process"]}
- ปัญหาที่เลือก: {mock_risk_map["risk"]}
- หน่วยที่ได้รับผลกระทบ: {mock_risk_map["affected_unit"]}

Input Block:
{mock_input}

Check Condition Block:
- ห้ามสรุป Root Cause เกินหลักฐาน
- ห้ามเสนอ Release, quarantine หรือ Recall โดยไม่มี Human approval
- ใช้เฉพาะข้อมูลจริงที่มีให้
- ถ้าข้อมูลไม่พอ ให้ระบุว่า "ยังสรุปไม่ได้"
- ย้อนกลับไปที่ lot, batch, line, shift หรือเวลาได้

งานที่ต้องการให้วิเคราะห์:
1. ระบุ signal ที่สำคัญ
2. ตั้งสมมติฐานที่เป็นไปได้
3. ระบุหลักฐานที่สนับสนุนแต่ละสมมติฐาน
4. ระบุหลักฐานที่ยังขาด
5. เสนอคำถามที่ควรถามทีม Production หรือ Quality Assurance เพิ่ม

Output Format Structure:
{{
  "signal_summary": "...",
  "hypotheses": [
    {{
      "hypothesis": "...",
      "supporting_evidence": ["..."],
      "missing_evidence": ["..."]
    }}
  ],
  "follow_up_questions": ["..."]
}}
""".strip()

analysis_from_AI = default_llm.invoke(improved_iris_analysis_prompt)

# IMPROVED VALIDATION PROMPT

improved_iris_validation_prompt = f"""
คุณคือผู้ช่วยตรวจสอบผลการวิเคราะห์ในอุตสาหกรรมอาหาร

Context Block:
- กระบวนการ: {mock_risk_map["process"]}
- ปัญหาที่เลือก: {mock_risk_map["risk"]}
- หน่วยที่ได้รับผลกระทบ: {mock_risk_map["affected_unit"]}

Input Block:
- AI-generated analysis: {analysis_from_AI.content}
- Decision constraints: {json.dumps(mock_risk_map["decision_constraints"], ensure_ascii=False, indent=2)}

Task:
1. คำตอบอ้างอิงข้อมูลจริงหรือไม่
2. ย้อนกลับไปที่ lot, batch, line, shift หรือเวลาได้ไหม
3. มีสมมติฐานใดที่ยังไม่มีหลักฐานสนับสนุน
4. มีความเสี่ยงด้าน food safety, compliance หรือ customer impact หรือไม่
5. จุดใดต้องให้มนุษย์อนุมัติก่อนดำเนินการ

Check Condition Block:
- หากหลักฐานไม่พอ ให้ระบุว่า “ยังสรุปไม่ได้” และบอกว่าต้องใช้ข้อมูลอะไรเพิ่ม

Output Format Structure:
{{
  "evidence_reference_check": {{
    "has_reference": TRUE/FALSE,
    "rationale": "..."
  }},
  "traceability_check": {{
    "lot": {{
      "found": TRUE/FALSE,
      "mentioned_info": "... or null"
    }},
    "batch": {{
      "found": TRUE/FALSE,
      "mentioned_info": "... or null"
    }},
    "line": {{
      "found": true,
      "mentioned_info": "... or null"
    }},
    "shift": {{
      "found": true,
      "mentioned_info": "... or null"
    }},
    "time": {{
      "found": true,
      "mentioned_info": "... or null"
    }}
  }},
  "unsupported_hypotheses": [
    {{
      "hypothesis": "...",
      "rationale": "..."
    }}
  ],
  "risk_check": {{
    "food_safety": {{
      "present": TRUE/FALSE,
      "rationale": "..."
    }},
    "compliance": {{
      "present": TRUE/FALSE,
      "rationale": "..."
    }},
    "customer_impact": {{
      "present": TRUE/FALSE,
      "rationale": "..."
    }}
  }},
  "human_decision_check": {{
    "has_human_decision_point": TRUE/FALSE,
    "detail": [
      "..."
    ]
  }},
  "needs_more_data": {{
    "required": TRUE/FALSE,
    "missing_data": [
      "..."
    ],
    "summary": "..."
  }},
  "validation_summary": "..."
}}
""".strip()

validation_from_AI = default_llm.invoke(improved_iris_validation_prompt)

print("=== IRIS ANALYSIS OUTPUT ===")
print(analysis_from_AI.content)
print()
print("=== IRIS VALIDATION OUTPUT ===")
print(validation_from_AI.content)


=== IRIS ANALYSIS OUTPUT ===
{
  "signal_summary": "สัญญาณสำคัญในการดำเนินการ cooling หลังสุก: batch B-0519-N บน Line 2 (Night shift) พบ deviation ใน cooling profile ตั้งแต่ช่วงต้นจนถึงภายหลัง โดยระบุว่าเริ่มเย็นจาก 60°C แล้วไม่สามารถถึง 21°C ภายใน 2 ชั่วโมง (02:00 บันทึก 24.8°C) ต่อด้วยการลดลงเหลือ 14.2°C ใน 03:00 และถึง 5.7°C ใน 06:00 อย่างไรก็ตาม batch ถูกโหลดเข้าแชลเลอร์ช้ากว่าแผน 20 นาที ความผิดปกติของประตูชิลเลอร์เปิดในเวลา 01:15 ถูกบันทึกใน log แม้ว่าระยะเวลาของ alarm ไม่ถูกระบุ QC ระบุไม่มีข้อบกพร่องของแพ็กเกจและโลตโค้ดถูกตรวจสอบ ส่วนบรรทัดเดิมที่เคยผลิต 3 บทก่อนหน้านี้ไม่มี deviation และ QA ต้องอิงการอนุมัติจาก Food Safety owner สำหรับการปล่อย/กักกัน/เรียกคืน"
,
  "hypotheses": [
    {
      "hypothesis": "ความล่าช้าในการโหลดเข้าแชลเลอร์ (ตามบันทึกการทำงานที่ระบุว่าโหลดชิลเลอร์ช้ากว่าแผน 20 นาที) อาจทำให้เวลาการคูลลิ่งจริงผิดแผน",
      "supporting_evidence": [
        "operator note: batch loaded into the chiller 20 minutes later than planned",
        "02:00 temperature = 24

## 7. Wrap-up

Across the lab:
- Food QA Behavior and Rules (`Demo 1a`)
- Shop-floor note -> risk context with ICIO `(DEMO 1b)`
- Diagnostic content -> structured Signal Map JSON `(DEMO 2a)`
- Check conditions -> JSON `null` / Python `None` `(DEMO 2b)`
- JSON prompt -> Pydantic schema `(DEMO 2c)`
- Human questions -> CoT-style diagnostic trace `(DEMO 3a)`
- Zero-shot narrative -> few-shot narrative pattern `(DEMO 3b)`
- Few-shot CoT reasoning trace -> reasoning pattern by example `(DEMO 3c)`
- Model Behavior -- Temperature Setting: write alternative internal email subject lines and observe output stability/diversity `(DEMO 4a)`
- Model Behavior -- Reasoning Effort Setting: verify statements with PASS / FAIL / UNKNOWN and observe evidence gating `(DEMO 4b)`
- IRIS-style analysis / validation flow `(Take-home)`
